In [ ]:
import sys
from pathlib import Path

import hydra
import joblib
import numpy as np
import pandas as pd
from omegaconf import DictConfig

from data_handler import DataHandler
from probes.conformal import MulticlassICP
from utils import _atomic_joblib_dump, _atomic_write_json, log_metric_multiclass
from utils_jupyter import load_hydra_config_with_params

Given the scores from the zero-shot prompting, we compute the metric.
1. Location of scores: `outputs/probes/prompt/default`
2. Output location for the metric: `outputs/probes/zero_shot`

**Note**: Move this notebook outside this folder to make it work


In [ ]:
models = ['qwen-2.5-14b',
          'qwen-2.5-7b',
          'mistral-7B-v0.3',
          'gemma-2-9b',
          'gemma-7b',
          'llama-3-8b',
          'llama-3.2-3b',
          '_qwen-2.5-14b',
          '_qwen-2.5-7b',
          '_mistral-7B-v0.3',
          '_gemma-2-9b',
          '_gemma-7b',
          '_llama-3.1-8b',
          '_llama-3-8b-med',
          '_llama-3.1-8b-bio',
          '_llama-3.2-3b']
    
datapacks = ['cities_loc', 'med_indications', 'defs']

In [ ]:
def save(concept_direction: np.ndarray,
         concept_bias: np.ndarray,
         metric_dict: dict,
         layer_id: int,
         cfg: DictConfig,
         estimator:   None,
         conf_calibrator:   None,
         scaler:  None,
         transformer: None,
         y_hat: np.ndarray = None, y_true: np.ndarray = None):
    '''
    Save the artifacts of the run.
    Args:
        concept_direction: The direction of the concept (coef_).
        concept_bias: The bias of the concept (intercept_).
        scaler: The scaler used for the features.
        transformer: The transformer used for the features.
        conf_calibrator: The conformal calibrator used for the predictions.
        metric_dict: The dictionary containing the metrics.
        cfg: The configuration object.
        layer_id: The ID of the layer.
        y_hat: The predicted labels.
        y_true: The true labels.
        estimator: The object of the classifier/regressor.

    '''
    output_dir = Path("outputs/probes/zero_shot/") / cfg.model['name'] / cfg.datapack['name']
    output_dir.mkdir(parents=True, exist_ok=True)

    # 1. Config (resolved)
    # 2. Metrics
    metrics_path = output_dir / "metrics.json"
    _atomic_write_json(metrics_path, metric_dict)
    # 3. Objects
    est_path = output_dir / \
        "estimator.joblib" if estimator is not None else None
    if (estimator is not None):
        _atomic_joblib_dump(est_path, estimator)
    scl_path = output_dir / \
        "scaler.joblib" if scaler is not None else None
    if (scaler is not None):
        _atomic_joblib_dump(scl_path, scaler)
    trn_path = output_dir / \
        "transformer.joblib" if transformer is not None else None
    if (transformer is not None):
        _atomic_joblib_dump(trn_path, transformer)
    clb_path = output_dir / \
        "calibrator.joblib" if conf_calibrator is not None else None
    if (conf_calibrator is not None):
        _atomic_joblib_dump(clb_path, conf_calibrator)

    coef_path = output_dir / \
        "coef.npy" if concept_direction is not None else None
    bias_path = output_dir / \
        "bias.npy" if concept_bias is not None else None
    # 4. Numpy arrays
    if concept_direction is not None:
        np.save(coef_path, concept_direction)
    if concept_bias is not None:
        np.save(bias_path, concept_bias)

    yh_path = output_dir / \
        "y_hat.npy" if y_hat is not None else None
    if (y_hat is not None):
        np.save(yh_path, y_hat)
    else:
        log.warning("y_hat is None")
    yt_path = output_dir / "y_true.npy" if y_true is not None else None
    if (y_true is not None):
        np.save(yt_path, y_true)

        # 6) Manifest (quick glance + reproducibility bits)
    manifest = {
        "type": "zero_shot",
        "paths": {
            "config": None,
            "metrics": str(metrics_path),
            "coef": None,
            "bias": None,
            "preds": str(yh_path) if (y_hat is not None or y_true is not None) else None,
            "estimator": None,
            "scaler": None,
            "transformer": None,
            "calibrator": str(clb_path) if clb_path else None,
        },
        "shapes": {
            "coef": None,
            "bias": None,
            "y_hat": None if y_hat is None else tuple(np.shape(y_hat)),
            "y_true": None if y_true is None else tuple(np.shape(y_true)),
        },
        "dtypes": {
            "coef": None,
            "bias": None,
            "y_hat": None if y_hat is None else str(getattr(y_hat, "dtype", "")),
            "y_true": None if y_true is None else str(getattr(y_true, "dtype", "")),
        },
        "env": {
            "python": sys.version.split()[0],
            "numpy": np.__version__,
            "joblib": joblib.__version__,
            "hydra": hydra.__version__,
            "sklearn": sys.modules['sklearn'].__version__,
            "scipy": sys.modules['scipy'].__version__,
            "polars": sys.modules['polars'].__version__ if 'polars' in sys.modules else "",
            "pandas": sys.modules['pandas'].__version__ if 'pandas' in sys.modules else "",
            "torch": sys.modules['torch'].__version__ if 'torch' in sys.modules else "",
        },
    }
    manifest_path = output_dir / "manifests"
    manifest_path.mkdir(parents=True, exist_ok=True)
    manifest_path = manifest_path / "manifest.json"
    _atomic_write_json(manifest_path, manifest)
    return manifest

def get_zero_shot_probs(probs: np.ndarray) -> np.ndarray:
    """
    Return the zero-shot probabilities
    Args:
        probs (np.ndarray): The probabilities from the model, shape (n_samples, 6)
    Returns:
        np.ndarray: The zero-shot probabilities, shape (n_samples, 4)
    """
    
    preds = np.zeros(shape=(probs.shape[0], 4))
    preds[:,0] = probs[:, 1] # False
    preds[:,1] = probs[:, 0] # True
    preds[:,2] = probs[:, 2] + probs[:, 3] # IDK
    preds[:,3] = probs[:, 4:].sum(axis=1) # Anstain
    return preds

def zero_shot_probs_from_df(df: pd.DataFrame):
    """
    Return the zero-shot probabilities from a dataframe
    """
    probs = df[[f'scores_{i}' for i in range(6)]].to_numpy()
    try:
        labels = df['multiclass_label'].values
    except:
        labels = np.zeros(shape=(probs.shape[0],))
        labels[df['correct'] == 1] = 1
        labels[df['fake_object'] == 1] = 2
    return get_zero_shot_probs(probs), labels


In [ ]:
for model in models:
    for datapack in datapacks:
        cfg = load_hydra_config_with_params(model = model, datapack=datapack, task=1, probe='mean_diff', config_name='probe_training') # task and probe does not really matter here

        dh = DataHandler(model=cfg.model["name"], datasets=cfg.datapack["datasets"],
                            dataset_path=cfg.setup["dataset_path"], activations_path=cfg.setup["activations_path"],
                            output_path=cfg.setup["output_path"],
                            activation_type=cfg.agg, with_calibration=True, load_scores='default',
                            )
        dh.assemble(test_size=cfg.datapack["test_size"], calibration_size=cfg.datapack["cal_size"],
                        seed=cfg.datapack["random_seed"], exclusive_split=cfg.datapack["exclusive_split"])

        probs, labels = zero_shot_probs_from_df(dh.get_cal_df())
        preds = np.argmax(probs, axis=1)
        cnf = MulticlassICP(n_classes=4)
        cnf.fit(y=labels, scores=probs)
        init_probs_te , labels_te = zero_shot_probs_from_df(dh.get_test_df())
        init_preds_te = np.argmax(init_probs_te, axis=1)
        init_preds_te[init_preds_te == 3] = -1  # Map 'Abstain' to -1
        conf_preds_te = cnf.predict(scores=init_probs_te)

        metric_dict = {}
        metric_dict['default'] = log_metric_multiclass(preds = init_preds_te, scores = init_probs_te, y_true=labels_te, mask=np.ones_like(labels_te), cfg=cfg)
        metric_dict['conformal'] = log_metric_multiclass(preds = conf_preds_te, scores = init_probs_te, y_true=labels_te, mask=np.ones_like(labels_te), cfg=cfg)


        save(concept_direction=None,
            concept_bias=None,
            metric_dict=metric_dict,
            layer_id=None, 
            cfg=cfg,
            estimator=None,
            conf_calibrator=cnf,
            scaler=None,
            transformer=None,
            y_hat=init_probs_te,
            y_true=labels_te)